# 01 — Environnement et exploration des données
Projet **Classification multilingue basée sur un LLM pour un service support** (formation LLM Engineer).
Ce notebook vérifie l'environnement (DGX Spark), charge **uniquement les 5 colonnes autorisées** et examine ce qui conditionne le résultat : déséquilibre des classes, langues, **ambiguïté des étiquettes** et **quasi-doublons** (y compris traduits).

In [1]:
import sys, os, json, time, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
from classif import config, data, prompts, evaluation
config.RESULTS_DIR.mkdir(exist_ok=True)

## 1. Environnement : GPU, bibliothèques, modèles

In [2]:
import subprocess, torch, transformers, peft, sklearn
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(smi)
(config.BASE_DIR / "evidence").mkdir(exist_ok=True)
(config.BASE_DIR / "evidence" / "nvidia-smi.txt").write_text(smi)
print("torch", torch.__version__, "| CUDA dispo :", torch.cuda.is_available(), "| transformers", transformers.__version__, "| peft", peft.__version__)
from classif import vllm_client
print("LLM open weights à personnaliser :", config.BASE_MODEL)
print("LLM de référence servi par vLLM  :", vllm_client.model_name())

Tue Oct 20 15:28:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GB10                    On  |   0000000F:01:00.0 Off |                  N/A |
| N/A   54C    P0             13W /  N/A  | Not Supported          |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

LLM open weights à personnaliser : Qwen/Qwen3-8B
LLM de référence servi par vLLM  : unsloth/Qwen3.8-27B-NVFP4


## 2. Chargement : seules les colonnes autorisées sont lues
L'énoncé interdit d'utiliser `answer`, `type`, `priority`, `tag_1…tag_9` (inconnus au moment de la classification réelle). `data.load_tickets` utilise `usecols` : ces colonnes ne sont jamais chargées en mémoire.

In [3]:
toutes = pd.read_csv(config.TICKETS_CSV, nrows=0).columns.tolist()
df = data.load_tickets()
print("colonnes du CSV  :", toutes)
print("colonnes utilisées:", list(df.columns))
print("colonnes ignorées :", [c for c in toutes if c not in df.columns])
print(df.shape, "| objets vides :", int((df.subject == '').sum()), "| messages vides :", int((df.body == '').sum()))
df.head(3)

colonnes du CSV  : ['id', 'subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'business_type', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9']
colonnes utilisées: ['subject', 'body', 'queue', 'language', 'business_type']
colonnes ignorées : ['id', 'answer', 'type', 'priority', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9']
(600, 5) | objets vides : 1 | messages vides : 0


,subject,body,queue,language,business_type
ticket,,,,,
0,Anfrage zu den Spezifikationen und Anpassungsoptionen des MacBook Air M1,"Sehr geehrtes Support-Team des Tech Online Stores,\n\nich interessiere mich für den Ka...",Customer Service,de,Tech Online Store
1,Déconnexions fréquentes et plantages,Le client signale des déconnexions fréquentes et des plantages lors des réunions vidéo...,Product Support,fr,Software Development Company
2,Problema de sonido Dell XPS,"Problema con el sonido, manejando como devolución. Dentro de 30 días.",Returns and Exchanges,es,Tech Online Store


## 3. Distribution des classes, des langues et des secteurs

In [4]:
cl = df.queue.value_counts().rename("n").to_frame()
cl["part"] = (cl.n / cl.n.sum()).round(3)
display(cl)
display(pd.crosstab(df.queue, df.language, margins=True))
display(pd.crosstab(df.queue, df.business_type, margins=True))
L = (df.subject.str.len() + df.body.str.len())
print("longueur subject+body (caractères) :", L.describe().round(0).to_dict())

,n,part
queue,,
Technical Support,210,0.350
Product Support,93,0.155
Customer Service,85,0.142
IT Support,77,0.128
Billing and Payments,46,0.077
Returns and Exchanges,41,0.068
Human Resources,15,0.025
Service Outages and Maintenance,15,0.025
Sales and Pre-Sales,13,0.022


language,de,en,es,fr,pt,All
queue,,,,,,
Billing and Payments,6,12,14,7,7,46
Customer Service,24,22,21,8,10,85
General Inquiry,4,0,0,0,1,5
Human Resources,1,4,5,3,2,15
IT Support,16,19,19,7,16,77
Product Support,25,32,10,12,14,93
Returns and Exchanges,13,11,6,6,5,41
Sales and Pre-Sales,4,2,4,0,3,13
Service Outages and Maintenance,4,4,2,3,2,15


business_type,IT Consulting Firm,IT Services,Software Development Company,Tech Online Store,All
queue,,,,,
Billing and Payments,0,23,4,19,46
Customer Service,5,22,11,47,85
General Inquiry,0,0,5,0,5
Human Resources,3,12,0,0,15
IT Support,11,56,4,6,77
Product Support,0,10,16,67,93
Returns and Exchanges,0,0,0,41,41
Sales and Pre-Sales,4,0,4,5,13
Service Outages and Maintenance,0,11,0,4,15


longueur subject+body (caractères) : {'count': 600.0, 'mean': 360.0, 'std': 133.0, 'min': 48.0, '25%': 285.0, '50%': 344.0, '75%': 432.0, 'max': 886.0}


**Lecture.** Dix classes très déséquilibrées : *Technical Support* pèse 35 % tandis que *General Inquiry* ne compte que 5 tickets (<1 %). D'où la métrique **F1 pondéré** et un **découpage stratifié**. Les cinq langues sont présentes dans presque toutes les classes.

## 4. Ambiguïté des étiquettes : un même sujet se retrouve dans plusieurs files

In [5]:
for mot in ["Jira", "Dell XPS", "Cisco", "Office 365", "Visual Studio Code"]:
    sub = df[df.subject.str.contains(mot, case=False) | df.body.str.contains(mot, case=False)]
    print(f"{mot:20s} n={len(sub):3d}", sub.queue.value_counts().to_dict())

Jira                 n= 51 {'Technical Support': 24, 'Product Support': 11, 'IT Support': 7, 'Customer Service': 6, 'Human Resources': 3}
Dell XPS             n=107 {'Technical Support': 43, 'Product Support': 26, 'Customer Service': 14, 'Returns and Exchanges': 13, 'IT Support': 11}
Cisco                n= 40 {'Technical Support': 21, 'IT Support': 19}
Office 365           n= 24 {'Billing and Payments': 7, 'Customer Service': 5, 'Sales and Pre-Sales': 4, 'Service Outages and Maintenance': 4, 'Technical Support': 2, 'Product Support': 2}
Visual Studio Code   n= 16 {'General Inquiry': 5, 'Human Resources': 4, 'IT Support': 4, 'Technical Support': 2, 'Product Support': 1}


In [6]:
pd.concat([df[df.queue == q].sample(2, random_state=1) for q in ["Human Resources", "Customer Service", "Product Support"]])[["queue", "language", "business_type", "subject"]]

,queue,language,business_type,subject
ticket,,,,
51,Human Resources,en,IT Services,High-Priority Assistance Needed for Visual Studio Code Issues
102,Human Resources,es,IT Consulting Firm,Urgente: Se necesita asistencia inmediata para la falla de la aplicación de nómina
288,Customer Service,fr,Tech Online Store,Urgent : Problème de temps de démarrage lent sur MacBook Air M1
305,Customer Service,fr,Tech Online Store,Diagnostics matériels immédiats pour Dell XPS 13 9310
404,Product Support,en,Software Development Company,Request to Upgrade Jira
486,Product Support,es,Tech Online Store,Problema con Microsoft Office 365: Excel


**Lecture.** Les files ne se déduisent pas du produit ou du vocabulaire seul (un ticket « Visual Studio Code » peut finir en *Human Resources*). Une partie du bruit vient des opérateurs humains. On s'attend donc à un plafond sous 100 % pour tout classifieur, et il faut le mesurer.

## 5. Point de repère : un classifieur classique (TF-IDF + régression logistique)
Validation croisée stratifiée à 5 plis sur les 600 tickets (aucun LLM).

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
X = df.business_type + " | " + df.language + " | " + df.subject + "\n" + df.body
rows = []
for nom, vec in [("mots 1-2", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
                 ("caractères 2-5", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True))]:
    pred = cross_val_predict(make_pipeline(vec, LogisticRegression(C=30, max_iter=3000)), X, df.queue,
                             cv=StratifiedKFold(5, shuffle=True, random_state=0))
    rows.append({"features": nom, "F1 pondéré (CV 5 plis)": round(f1_score(df.queue, pred, average="weighted"), 3),
                 "exactitude": round((pred == df.queue).mean(), 3)})
pd.DataFrame(rows)

,features,F1 pondéré (CV 5 plis),exactitude
0,mots 1-2,0.569,0.585
1,caractères 2-5,0.674,0.682


## 6. Quasi-doublons, y compris traduits
Le jeu de 600 demandes est issu de modèles de messages déclinés en plusieurs langues et en reformulations. On cherche, pour chaque ticket, le plus proche voisin avec un encodeur **multilingue** (`BAAI/bge-m3`), puis on regarde si le voisin porte la même file. Si oui, un test tiré au hasard contient des tickets dont une « copie » figure dans l'entraînement : cela pèse sur l'interprétation du score (cf. notebook 04).

In [8]:
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer("BAAI/bge-m3", device="cuda")
E = enc.encode((df.subject + "\n" + df.body).tolist(), batch_size=32, normalize_embeddings=True, show_progress_bar=False)
S = E @ E.T; np.fill_diagonal(S, -1)
voisin, sim = S.argmax(1), S.max(1)
meme = (df.queue.values == df.queue.values[voisin])
lang_diff = (df.language.values != df.language.values[voisin])
lignes = []
for t in [0.95, 0.9, 0.85, 0.8, 0.7, 0.0]:
    k = sim >= t
    lignes.append({"similarité du plus proche voisin ≥": t, "tickets": int(k.sum()), "part des 600": round(k.mean(), 3),
                   "même file que le voisin": round(meme[k].mean(), 3), "voisin dans une autre langue": round(lang_diff[k].mean(), 3)})
display(pd.DataFrame(lignes))
pd.DataFrame({"voisin": voisin, "similarite": sim.round(4), "meme_queue": meme}).to_csv(config.RESULTS_DIR / "quasi_doublons.csv")
del enc, E, S; torch.cuda.empty_cache()

,similarité du plus proche voisin ≥,tickets,part des 600,même file que le voisin,voisin dans une autre langue
0,0.95,209,0.348,1.000,0.368
1,0.90,486,0.810,0.984,0.650
2,0.85,577,0.962,0.977,0.693
3,0.80,595,0.992,0.978,0.703
4,0.70,600,1.000,0.977,0.705
5,0.00,600,1.000,0.977,0.705


**Lecture.** Beaucoup de tickets ont un voisin très proche, souvent dans une autre langue, et la file du voisin est quasi toujours la même. Un modèle qui a vu un ticket pendant l'entraînement reconnaît donc sa traduction ou sa reformulation dans le test : c'est légitime pour l'énoncé (test stratifié tiré au hasard) mais le score obtenu ne mesure pas la même chose que la généralisation à des demandes inédites. Le notebook 04 rapporte **les deux**.